# CSV Chunking

## 1. Direct Vector Embedding Approach
- Process: Convert CSV rows/columns → Text chunks → Vector embeddings → Vector database
- Best for: Smaller datasets with clear semantic meaning
- Tools: LangChain's CSVLoader, LlamaIndex's CSVNodeParser
- Advantage: Simplest implementation with minimal preprocessing
## 2. CSV to JSON Conversion
- Process: CSV → JSON → Text chunks → Vector embeddings → Vector database
- Best for: Complex CSV structures with nested relationships
- Tools: Pandas for conversion, any RAG framework for processing
- Advantage: Better preservation of data relationships and structure
## 3. Structured Query Approach
- Process: CSV → Pandas DataFrame → LLM-generated SQL-like queries → Results to LLM
- Best for: Analytical questions requiring precise data manipulation
- Tools: Pandas, LangChain agents
- Advantage: Maintains data integrity, excellent for numerical analysis
## 4. Hybrid Chunking Strategy
- Process: CSV → Multiple chunking strategies → Combined vector database
- Best for: Complex datasets with varied query patterns
- Tools: LlamaIndex, Haystack
- Advantage: Balances between semantic search and structured data integrity
## 5. Summarization-First Approach
- Process: CSV → LLM summarization → Vector embeddings of summaries → Original data retrieval
- Best for: Very large CSV files with clear thematic sections
- Tools: Any LLM for summarization, vector database for storage
- Advantage: Reduces storage requirements while maintaining context

## 1. Direct Vector Embedding Approach

In [1]:
import pandas as pd

df = pd.read_csv("customer.csv")

def row_to_text(row):
    return (
        f"Customer ID {row['customer_id']} "
        f"is {row['name']} from {row['city']}. "
        f"Age is {row['age']} years. "
        f"Purchase amount is ${row['purchase_amount']}."
    )

documents = df.apply(row_to_text, axis=1).tolist()



In [2]:
documents

['Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.',
 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.',
 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.']

In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11832.50it/s]


In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="customers",
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    )
)

True

In [6]:
points = []

for idx, (text, vector) in enumerate(zip(documents, embeddings)):

    points.append(
        PointStruct(
            id=idx,
            vector=vector.tolist(),
            payload={
                "text": text,
                "customer_id": int(df.iloc[idx]["customer_id"]),
                "name": str(df.iloc[idx]["name"]),
                "city": str(df.iloc[idx]["city"]),
                "age": int(df.iloc[idx]["age"]),
                "purchase_amount": float(
                    df.iloc[idx]["purchase_amount"]
                )
            }
        )
    )

client.upsert(
    collection_name="customers",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
query = "customers with high purchases"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

In [10]:
results = client.query_points(
    collection_name="customers",
    query=query_embedding.tolist(),
    limit=3
)

print(results)

points=[ScoredPoint(id=2, version=0, score=0.6109242454756989, payload={'text': 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.', 'customer_id': 103, 'name': 'Bob', 'city': 'Boston', 'age': 42, 'purchase_amount': 800.0}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.5949853675040405, payload={'text': 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.', 'customer_id': 102, 'name': 'Alice', 'city': 'Chicago', 'age': 35, 'purchase_amount': 1200.0}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=0, version=0, score=0.5748373312834694, payload={'text': 'Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.', 'customer_id': 101, 'name': 'John', 'city': 'New York', 'age': 28, 'purchase_amount': 500.0}, vector=None, shard_key=None, order_value=None)]


In [11]:
for hit in results.points:
    print(hit.payload)

{'text': 'Customer ID 103 is Bob from Boston. Age is 42 years. Purchase amount is $800.', 'customer_id': 103, 'name': 'Bob', 'city': 'Boston', 'age': 42, 'purchase_amount': 800.0}
{'text': 'Customer ID 102 is Alice from Chicago. Age is 35 years. Purchase amount is $1200.', 'customer_id': 102, 'name': 'Alice', 'city': 'Chicago', 'age': 35, 'purchase_amount': 1200.0}
{'text': 'Customer ID 101 is John from New York. Age is 28 years. Purchase amount is $500.', 'customer_id': 101, 'name': 'John', 'city': 'New York', 'age': 28, 'purchase_amount': 500.0}
